# Transit Route Generation Evaluation

This notebook evaluates two approaches for transit network design:
- **LC (Learning-based Construction)** – pure neural route generator
- **Neural BCO (Bee Colony Optimization with neural initialization)**

Tested on **Mandl's benchmark** and **Tartu real-world network**.

---

## 1. Setup & Configuration

In [ ]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import pickle
from pathlib import Path
from omegaconf import OmegaConf
from torch_geometric.loader import DataLoader
from tqdm import tqdm

# Project-specific imports
from connectpt.routes_generator.citygraph_dataset import get_dataset_from_config
from connectpt.routes_generator.utils import get_eval_cfg
from connectpt.routes_generator.eval_route_generator import eval_model
from connectpt.routes_generator.torch_utils import dump_routes
from connectpt.routes_generator import bee_colony
import connectpt.routes_generator.utils as lrnu

In [ ]:
# Root paths (adjust only here if needed)
ROOT_DIR = Path.cwd().parent.parent
DATA_DIR = ROOT_DIR / "examples/data"
CFG_DIR = ROOT_DIR / "connectpt/routes_generator/cfg"
MODEL_WEIGHTS_PATH = ROOT_DIR / "examples/data/model_weights/inductive_random_graphs_weighted_connectivity.pt"

# Verify paths
print("Root directory:", ROOT_DIR)
print("Config directory:", CFG_DIR)
print("Model weights exist:", MODEL_WEIGHTS_PATH.exists())

Root directory: /root/transport
Config directory: /root/transport/transport/routes_generator/cfg
Model weights exist: True


## 2. Load Benchmark Instances

### 2.1 Mandl's Benchmark (Classic 15-node instance)

In [ ]:
def load_mandl_instance():
    data_dir = DATA_DIR / "benchmark"
    instance_name = "Mandl"

    # Coordinates
    coords_path = data_dir / f"{instance_name}Coords.txt"
    node_locs = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)

    # Travel times (in seconds)
    tt_path = data_dir / f"{instance_name}TravelTimes.txt"
    street_adj = torch.tensor(np.genfromtxt(tt_path), dtype=torch.float32) * 60

    # Demand matrix
    dmd_path = data_dir / f"{instance_name}Demand.txt"
    demand = torch.tensor(np.genfromtxt(dmd_path), dtype=torch.float32)

    print(f"Mandl instance loaded:")
    print(f"  Nodes: {node_locs.shape[0]}, Demand shape: {demand.shape}")

    return {
        'street_adj': street_adj,
        'demand': demand,
        'node_locs': node_locs
    }

# Load Mandl
input_tensors_mandl = load_mandl_instance()

Mandl instance loaded:
  Nodes: 15, Demand shape: torch.Size([15, 15])


### 2.2 Tartu Real-World Network (199 nodes)

In [4]:
def load_tartu_instance():
    data_dir = DATA_DIR / "tartu"

    # Load graph
    with open(data_dir / "bus_graph_Tartu.pkl", "rb") as f:
        G = pickle.load(f)

    # Load demand (OD matrix)
    OD = pd.read_csv(data_dir / "OD_matrix_Tartu.csv", index_col=0)
    demand = torch.tensor(OD.values, dtype=torch.float32)

    # Sort nodes to match OD matrix order
    nodes = sorted(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    # Node coordinates
    node_locs = torch.tensor([[G.nodes[n]['x'], G.nodes[n]['y']] for n in nodes], dtype=torch.float32)

    # Build symmetric travel time matrix
    n = len(nodes)
    inf = float('inf')
    adj_matrix = np.full((n, n), inf, dtype=np.float32)
    np.fill_diagonal(adj_matrix, 0.0)

    for u, v, data in G.edges(data=True):
        if 'time_min' in data:
            t = data['time_min']
            i, j = node_to_idx[u], node_to_idx[v]
            adj_matrix[i, j] = t
            adj_matrix[j, i] = t

    street_adj = torch.tensor(adj_matrix)
    street_adj[street_adj == inf] = torch.inf

    print(f"Tartu instance loaded: {n} nodes")

    return {
        'street_adj': street_adj,
        'demand': demand,
        'node_locs': node_locs
    }

# Uncomment to use Tartu instead
# input_tensors = load_tartu_instance()
input_tensors = input_tensors_mandl  # Default: Mandl

## 3. Evaluation Parameters

In [5]:
eval_params = {
    "dataset_name": "tensor",
    "n_routes": 6,
    "min_route_len": 2,
    "max_route_len": 8,
    "demand_time_weight": 0.33,
    "route_time_weight": 0.33,
    "median_connectivity_weight": 0.33,
    "run_name": "mandl_eval",
    "model_weights": str(MODEL_WEIGHTS_PATH)
}

## 4. Method 1: Learning-based Construction (LC)

In [8]:
# Load config
cfg_lc = get_eval_cfg(
    cfg_dir=str(CFG_DIR),
    base_cfg_name="eval_model_mumford",
    params=eval_params
)

# Build dataset and dataloader
test_ds = get_dataset_from_config(cfg_lc.eval.dataset, tensors=input_tensors)
test_dl = DataLoader(test_ds, batch_size=cfg_lc.batch_size)

# Initialize model
DEVICE, run_name_lc, _, cost_obj, model_lc = lrnu.process_standard_experiment_cfg(
    cfg_lc, run_name_prefix="lc_", weights_required=True
)

print(f"Running LC on {DEVICE}...")

_, unserved_lc, metrics_lc, routes_lc = eval_model(
    model_lc,
    test_dl,
    cfg_lc.eval,
    cost_obj,
    n_samples=cfg_lc.get("n_samples", 1),
    return_routes=True,
    silent=True,
    device=DEVICE
)

dump_routes(run_name_lc, routes_lc.cpu())

print("\nLC Results")
print("-" * 50)
print(f"Cost: {metrics_lc['cost'].item():.4f}")
print(f"Unserved demand (%): {metrics_lc['$d_{un}$'].item():.2f}")
print(f"Average Travel Time (ATT): {metrics_lc['ATT'].item():.4f}")
print(f"Routes:\n{routes_lc}")

/root/transport/transport/routes_generator/utils.py:171: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg.model.weights,


Running LC on cpu...

LC Results
--------------------------------------------------
Cost: 0.4384
Unserved demand (%): 0.06
Average Travel Time (ATT): 13.8658
Routes:
tensor([[[10,  9,  7,  5,  2,  1,  0],
         [ 8, 14,  6,  9, 12, -1, -1],
         [ 7,  5,  3, 11, -1, -1, -1],
         [12, 13, -1, -1, -1, -1, -1],
         [ 2,  1,  4, -1, -1, -1, -1],
         [ 3,  4, -1, -1, -1, -1, -1]]])


/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


## 5. Method 2: Neural Bee Colony Optimization (Neural BCO)

In [10]:
# Use same parameters but switch to BCO config
bco_params = eval_params.copy()
bco_params.update({
    "run_name": "neural_bco_mandl",
    "starting_routes_init": "tensor"  # Use LC routes as warm-start
})

cfg_bco = get_eval_cfg(
    cfg_dir=str(CFG_DIR),
    base_cfg_name="neural_bco_mumford",
    params=bco_params
)

# Re-use same dataset
test_dl_bco = DataLoader(test_ds, batch_size=cfg_bco.batch_size)

# Initialize bee colony (with optional neural guidance)
use_neural_bees = cfg_bco.get("neural_bees", True)
prefix = "neural_bco_" if use_neural_bees else "bco_"

DEVICE, run_name_bco, sum_writer, cost_obj_bco, bee_model = lrnu.process_standard_experiment_cfg(
    cfg_bco, prefix, weights_required=True
)

if bee_model is not None:
    bee_model.eval()

print(f"Running Neural BCO (warm-started from LC)...")

test_output = lrnu.test_method(
    bee_colony,
    test_dl_bco,
    cfg_bco.eval,
    cfg_bco.init,
    cost_obj_bco,
    n_bees=cfg_bco.n_bees,
    n_iterations=cfg_bco.n_iterations,
    device=DEVICE,
    bee_model=bee_model,
    return_routes=True,
    routes_tensor=routes_lc,  # Warm-start!
    silent=True
)

routes_bco = test_output[-1]
metrics_bco = test_output[-2]
unserved_bco = test_output[-3]

dump_routes(run_name_bco, routes_bco)

print("\nNeural BCO Results")
print("-" * 50)
print(f"Cost: {metrics_bco['cost'].item():.4f}")
print(f"Unserved demand (%): {metrics_bco['$d_{un}$'].item():.2f}")
print(f"Average Travel Time (ATT): {metrics_bco['ATT'].item():.4f}")
print(f"Improvement vs LC: {(metrics_lc['cost'] - metrics_bco['cost']).item():.4f}")
print(f"Final routes:\n{routes_bco}")

/root/transport/transport/routes_generator/utils.py:171: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg.model.weights,


Running Neural BCO (warm-started from LC)...

Neural BCO Results
--------------------------------------------------
Cost: 0.3861
Unserved demand (%): 0.71
Average Travel Time (ATT): 14.7341
Improvement vs LC: 0.0523
Final routes:
[[tensor([10,  9,  7,  5,  2,  1,  0]), tensor([ 6, 14,  8]), tensor([ 7,  5,  3, 11]), tensor([ 7, 14]), tensor([10, 12, 13]), tensor([3, 4])]]


/root/transport/transport/routes_generator/utils.py:319: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at ../aten/src/ATen/native/ReduceOps.cpp:1823.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


## 6. Summary Comparison

In [12]:
### 6. Summary Comparison (fixed)

print("=" * 60)
print("Final Comparison".center(60))
print("=" * 60)
print(f"{'Method':<15} {'Cost':<10} {'Unserved (%)':<15} {'ATT':<10} {'Time (s)':<10}")
print("-" * 60)

def _val(d, key, default=0.0):
    """Safely extract scalar from tensor or return default."""
    v = d.get(key, torch.tensor(default))
    return v.item() if hasattr(v, "item") else float(v)

# LC row
lc_cost   = _val(metrics_lc, 'cost')
lc_un     = _val(metrics_lc, '$d_{un}$')
lc_att    = _val(metrics_lc, 'ATT')
lc_time   = _val(metrics_lc, 'average wall-clock duration', 0)

# Neural BCO row
bco_cost  = _val(metrics_bco, 'cost')
bco_un    = _val(metrics_bco, '$d_{un}$')
bco_att   = _val(metrics_bco, 'ATT')
bco_time  = _val(metrics_bco, 'average wall-clock duration', 0)

print(f"{'LC':<15} {lc_cost:<10.4f} {lc_un:<15.2f} {lc_att:<10.4f} {lc_time:<10.1f}")
print(f"{'Neural BCO':<15} {bco_cost:<10.4f} {bco_un:<15.2f} {bco_att:<10.4f} {bco_time:<10.1f}")

# Optional: improvement highlight
if bco_cost < lc_cost:
    improvement = lc_cost - bco_cost
    print(f"\nNeural BCO improves cost by {improvement:.4f} ({improvement/lc_cost*100:.2f}% better)")

                      Final Comparison                      
Method          Cost       Unserved (%)    ATT        Time (s)  
------------------------------------------------------------
LC              0.4384     0.06            13.8658    0.3       
Neural BCO      0.3861     0.71            14.7341    1.4       

Neural BCO improves cost by 0.0523 (11.93% better)
